## Cell 1: Data Loading and Inspection

We load the Abalone dataset to predict age (Rings + 1.5). We must inspect the table structure to identify which columns are numeric and suitable for a linear model.

In [4]:
import pandas as pd
import numpy as np

# A1. Load dataset
# Ensure abalone.data or abalone.csv is in your directory
# Dataset columns: Sex, Length, Diameter, Height, Whole weight, Shucked weight,
# Viscera weight, Shell weight, Rings
columns = [
    "Sex", "Length", "Diameter", "Height",
    "Whole_weight", "Shucked_weight",
    "Viscera_weight", "Shell_weight", "Rings"
]
df = pd.read_csv("abalone.data", header=None, names=columns)

# Print dataset stats
print("Number of rows:", len(df))
print("Column names:", df.columns.tolist())
print(df.head())

# Checkpoint:
# what is input: Numeric physical measurements of abalone
# what is output: The 'Rings' column (to be converted to age)
# why output is numeric: Age is a continuous quantity, not a category

Number of rows: 4177
Column names: ['Sex', 'Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight', 'Rings']
  Sex  Length  Diameter  Height  Whole_weight  Shucked_weight  Viscera_weight  \
0   M   0.455     0.365   0.095        0.5140          0.2245          0.1010   
1   M   0.350     0.265   0.090        0.2255          0.0995          0.0485   
2   F   0.530     0.420   0.135        0.6770          0.2565          0.1415   
3   M   0.440     0.365   0.125        0.5160          0.2155          0.1140   
4   I   0.330     0.255   0.080        0.2050          0.0895          0.0395   

   Shell_weight  Rings  
0         0.150     15  
1         0.070      7  
2         0.210      9  
3         0.155     10  
4         0.055      7  


## Cell 2: Target Conversion and Feature Selection
 We convert the "Rings" value into a physical "Age" target. We then select exactly three numeric features to act as our inputs ($x_1, x_2, x_3$)

In [6]:
# A2. Convert target: y = Rings + 1.5
df['y'] = df['Rings'] + 1.5

# A3. Choose features (Pick 3 numeric features)
# Example: Length, Diameter, Whole_weight
features = ['Length', 'Diameter', 'Whole_weight']
X = df[features].values
y = df['y'].values.reshape(-1, 1) # Reshape for matrix multiplication

# Justification:
# Feature 1: Length - Primary indicator of growth
# Feature 2: Diameter - Correlates with age and shell thickness
# Feature 3: Whole weight - Overall mass typically increases with maturity

## Cell 3: Data Split and Normalization

We split the data into 80% training and 20% testing sets manually. We then normalize the inputs using the training set's mean and standard deviation to ensure stable learning.

In [7]:
# A4. Train-test split (80/20)
split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

# A5. Normalize inputs
mean = np.mean(X_train, axis=0)
std = np.std(X_train, axis=0)

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

# Checkpoint:
# why normalization is needed: Prevents features with large scales from
# dominating gradients and ensures the loss surface is easier to descend

X_train shape: (3341, 3)
X_test shape: (836, 3)


## Cell 4: Model and Loss Definition
We define the linear neuron's forward pass ($\hat{y} = Xw + b$) and the Mean Squared Error (MSE) function to quantify our mistakes.

In [8]:
# Part B: Define the model
def forward(X, w, b):
    # y_hat = Xw + b
    return X @ w + b

# Part C: Define loss (MSE)
def mse(y, y_hat):
    # L = (1/N) * sum((y - y_hat)^2)
    return np.mean((y - y_hat)**2)

# Checkpoint:
# why square: Removes negative signs and heavily penalizes large outliers
# what mistakes are expensive: Large errors (e.g., an error of 10 becomes 100)

## Cell 5: Gradient Implementation
 We implement functions to calculate the gradients for weights ($w$) and bias ($b$). The gradient represents the direction and slope of the loss increase.

In [9]:
# D3. Implement gradients
def grad_w(X, y, y_hat):
    # Error = y_hat - y
    error = y_hat - y
    # Gradient of MSE w.r.t w: (2/N) * X^T * error
    return (2 / len(y)) * (X.T @ error)

def grad_b(y, y_hat):
    error = y_hat - y
    # Gradient of MSE w.r.t b: (2/N) * sum(error)
    return (2 / len(y)) * np.sum(error)

# Checkpoint:
# what gradient means in words: The direction of steepest ascent for the loss
# why subtracting gradient reduces loss: It moves the parameters "downhill"

## Cell 6: The Training Loop
We initialize parameters and update them iteratively using the Gradient Descent rule: $param = param - (lr \times gradient)$.

In [10]:
# Part E: Training loop
# Initialize w with small random values, b with zero
w = np.random.randn(X_train.shape[1], 1) * 0.01
b = 0.0
lr = 0.1 # Learning rate
epochs = 100

for epoch in range(epochs):
    # 1) Forward pass
    y_hat = forward(X_train, w, b)

    # 2) Compute loss
    loss = mse(y_train, y_hat)

    # 3) Compute gradients
    dw = grad_w(X_train, y_train, y_hat)
    db = grad_b(y_train, y_hat)

    # 4) Update parameters
    w = w - lr * dw
    b = b - lr * db

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

Epoch 0, Loss: 144.2717
Epoch 10, Loss: 9.0104
Epoch 20, Loss: 7.4824
Epoch 30, Loss: 7.4557
Epoch 40, Loss: 7.4472
Epoch 50, Loss: 7.4396
Epoch 60, Loss: 7.4325
Epoch 70, Loss: 7.4260
Epoch 80, Loss: 7.4198
Epoch 90, Loss: 7.4141


## Cell 7: Evaluation and Reflective Comments

Finally, we evaluate the model on the test set using MSE and Mean Absolute Error (MAE) and inspect 5 specific predictions to look for systematic bias

In [11]:
# Part F: Evaluation
y_test_pred = forward(X_test, w, b)
test_mse = mse(y_test, y_test_pred)
test_mae = np.mean(np.abs(y_test - y_test_pred))

print(f"Test MSE: {test_mse:.4f}")
print(f"Test MAE: {test_mae:.4f}")

# Print 5 example predictions
print("\nTrue Age | Predicted Age | Absolute Error")
for i in range(5):
    true = y_test[i][0]
    pred = y_test_pred[i][0]
    err = abs(true - pred)
    print(f"{true:8.2f} | {pred:13.2f} | {err:14.2f}")

# Checkpoint:
# systematic errors: Errors often larger for very old or very young abalone
# observed bias: Linear models cannot bend to fit curved growth patterns

Test MSE: 5.3520
Test MAE: 1.8073

True Age | Predicted Age | Absolute Error
   13.50 |         11.06 |           2.44
   15.50 |          9.96 |           5.54
   14.50 |         10.50 |           4.00
   14.50 |         10.87 |           3.63
   13.50 |         10.91 |           2.59
